In [16]:
import pandas as pd
from transformers import pipeline
from tqdm.auto import tqdm

In [17]:
books = pd.read_csv("../data/processed/books_clean1.csv")

books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead
1,9780002261982,0002261987,Spider's Web,"Charles Osborne, Agatha Christie",Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain


In [18]:
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=0,
    top_k=None
)

Device set to use cuda:0


In [19]:
description = books.loc[0, "description"]

emotion_classifier(description)

[[{'label': 'fear', 'score': 0.6548418402671814},
  {'label': 'neutral', 'score': 0.16985197365283966},
  {'label': 'sadness', 'score': 0.11640845984220505},
  {'label': 'surprise', 'score': 0.020700769498944283},
  {'label': 'disgust', 'score': 0.01910068280994892},
  {'label': 'joy', 'score': 0.015161238610744476},
  {'label': 'anger', 'score': 0.0039351521991193295}]]

In [20]:
emotion, score = get_dominant_emotion(books.loc[0, "description"])

print(emotion, score)

fear 0.6548418402671814


In [21]:
from math import ceil

batch_size = 32

emotions = []
scores = []

descriptions = books["description"].tolist()

for i in tqdm(range(0, len(descriptions), batch_size)):
    batch = descriptions[i:i + batch_size]

    results = emotion_classifier(
        batch,
        truncation=True,
        max_length=510
    )

    for result in results:
        best = max(result, key=lambda x: x["score"])
        emotions.append(best["label"])
        scores.append(best["score"])

  0%|          | 0/163 [00:00<?, ?it/s]

In [22]:
books["dominant_emotion"] = emotions
books["emotion_score"] = scores

In [23]:
books[["title", "dominant_emotion", "emotion_score"]].head(10)

,title,dominant_emotion,emotion_score
0,Gilead,fear,0.654842
1,Spider's Web,fear,0.755521
2,Rage of angels,fear,0.939291
3,The Four Loves,anger,0.332783
4,The Problem of Pain,neutral,0.854798
5,Empires of the Monsoon,disgust,0.877142
6,The Gap Into Madness,neutral,0.472348
7,Master of the Game,sadness,0.564391
8,Warhost of Vastmark,sadness,0.282257
9,The Once and Future King,neutral,0.792439


In [24]:
books.sample(20)[
    ["title", "description", "dominant_emotion", "emotion_score"]
]

,title,description,dominant_emotion,emotion_score
468,The Confessions of Nat Turner,In 1831 Nat Turner awaits death in a Virginia ...,fear,0.587847
4574,Some Ether,"Winner of a ""Discovery""/The Nation Award Winne...",surprise,0.707750
1476,The King of Elfland's Daughter,A young prince ventures into a mysterious fore...,fear,0.758381
156,"Men Are from Mars, Women Are from Venus",Rediscover the most famous relationship book e...,neutral,0.890782
5081,Redwall,What can the peace-loving mice of Redwall Abbe...,neutral,0.560884
797,The Spy who Loved Me,James Bond has known quite a few women but the...,neutral,0.914071
854,The Big Over Easy,Unconvinced that a former convict and milliona...,fear,0.428685
4464,A Commentary Upon the Gospel According to S. Luke,This is a reproduction of the original artefac...,joy,0.955866
4362,Eat This Book,"Collects more than 150 recipes, ranging from a...",neutral,0.885683
716,Chocolat,When the beautiful and mysterious Vianne moves...,joy,0.985126


In [25]:
books.to_csv("../data/processed/books_with_emotions.csv", index=False)

In [26]:
books["categories"].nunique()

479

In [28]:
category_counts = books["categories"].value_counts()

category_counts.head(100)

categories
Fiction                      2111
Juvenile Fiction              390
Biography & Autobiography     311
History                       207
Literary Criticism            124
                             ... 
Australian fiction              2
Australia                       2
College teachers                2
Short stories, American         2
Horror tales                    2
Name: count, Length: 100, dtype: int64

In [29]:
books.head()


,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,dominant_emotion,emotion_score
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,fear,0.654842
1,9780002261982,0002261987,Spider's Web,"Charles Osborne, Agatha Christie",Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,fear,0.755521
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,fear,0.939291
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,anger,0.332783
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,neutral,0.854798
